# Whole-Genome Germline Variant Analysis Pipeline

## Germline Variant Calling Pipeline from BAM Files

### Workflow Overview

1. Download and Install Required Tools
2. Upload Input BAM Files
3. Download and Prepare the Reference Genome
4. Add Read Groups
5. Mark PCR Duplicates
6. Base Quality Score Recalibration (BQSR)
7. Germline Variant Discovery with GATK HaplotypeCaller
8. Extract SNPs and INDELs
9. Variant Filtration
10. Functional Annotation with SnpEff
11. Review Final Results

### Download and Install GATK (Genome Analysis Toolkit)

In [1]:
# 1. Download the latest stable GATK release (v4.6.2.0)
!wget https://github.com/broadinstitute/gatk/releases/download/4.6.2.0/gatk-4.6.2.0.zip

# 2. Unzip the package quietly
!unzip -q gatk-4.6.2.0.zip

# 3. Add the GATK directory to the system PATH so you can call 'gatk' directly from any cell
import os
os.environ['PATH'] += ':_ROOT_/content/gatk-4.6.2.0'
# For standard Colab environments, the absolute path is /content/gatk-4.6.2.0
os.environ['PATH'] += ':/content/gatk-4.6.2.0'

--2026-06-01 09:11:57--  https://github.com/broadinstitute/gatk/releases/download/4.6.2.0/gatk-4.6.2.0.zip
Resolving github.com (github.com)... 20.27.177.113
Connecting to github.com (github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/27452807/49d84b30-bde0-4eb1-95cf-fa3bf4636501?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-06-01T09%3A58%3A01Z&rscd=attachment%3B+filename%3Dgatk-4.6.2.0.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-06-01T08%3A57%3A50Z&ske=2026-06-01T09%3A58%3A01Z&sks=b&skv=2018-11-09&sig=O6MDrZQFYxyPLhH0%2BcdcGRJ%2B3Wtp6UA8GRhqRHv4cgw%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4MDMwODcxNywibmJmIjoxNzgwMzA1MTE3LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdG

In [2]:
# 4. Verify installation
!gatk --version

Using GATK jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar --version
The Genome Analysis Toolkit (GATK) v4.6.2.0
HTSJDK Version: 4.2.0
Picard Version: 3.4.0


### Install SAMtools for BAM File Processing

In [3]:
!apt-get install -y samtools

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libhts3 libhtscodecs2
Suggested packages:
  cwltool
The following NEW packages will be installed:
  libhts3 libhtscodecs2 samtools
0 upgraded, 3 newly installed, 0 to remove and 2 not upgraded.
Need to get 963 kB of archives.
After this operation, 2,270 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libhtscodecs2 amd64 1.1.1-3 [53.2 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libhts3 amd64 1.13+ds-2build1 [390 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 samtools amd64 1.13-4 [520 kB]
Fetched 963 kB in 0s (10.7 MB/s)
Selecting previously unselected package libhtscodecs2:amd64.
(Reading database ... 118242 files and directories currently installed.)
Preparing to unpack .../libhtscodecs2_1.1.1-3_amd64.deb ...
Unpacking libhtscodecs2:amd64 (1.1.1-

## Uploading and unziping Data (BAM file)

In [4]:
%%bash
# Use standard unzip to extract the files cleanly
unzip -qo /content/WES-LUNG.zip -d /content/

# Verify that the files are now successfully sitting in /content/WES-LUNG/
echo "📊 Checking extracted files:"
ls -lh /content/WES-LUNG/

📊 Checking extracted files:
total 16M
-rw-r--r-- 1 root root 6.0M Jan 21  2019 NORMAL.bam
-rw-r--r-- 1 root root 1.4M Jan 21  2019 NORMAL.bam.bai
-rw-r--r-- 1 root root 7.2M Jan 21  2019 TUMOR.bam
-rw-r--r-- 1 root root 1.4M Jan 21  2019 TUMOR.bam.bai


## Step 1: Download and Index Reference

In [5]:
%%bash
mkdir -p /content/reference_hg19
echo "📥 Downloading matching UCSC hg19 reference..."

# Download the hg19 fasta file
wget -q --show-progress -P /content/reference_hg19/ https://hgdownload.soe.ucsc.edu/goldenPath/hg19/bigZips/hg19.fa.gz
gunzip /content/reference_hg19/hg19.fa.gz

# Index the fasta file
echo "🔧 Indexing reference..."
samtools faidx /content/reference_hg19/hg19.fa

# Create the sequence dictionary
echo "🔧 Creating sequence dictionary..."
./gatk-4.6.2.0/gatk CreateSequenceDictionary -R /content/reference_hg19/hg19.fa

📥 Downloading matching UCSC hg19 reference...
🔧 Indexing reference...
🔧 Creating sequence dictionary...
Tool returned:
0



     0K .......... .......... .......... .......... ..........  0%  154K 1h40m
    50K .......... .......... .......... .......... ..........  0% 1.23M 56m17s
   100K .......... .......... .......... .......... ..........  0%  406K 50m11s
   150K .......... .......... .......... .......... ..........  0%  293M 37m39s
   200K .......... .......... .......... .......... ..........  0%  308K 40m8s
   250K .......... .......... .......... .......... ..........  0%  204M 33m27s
   300K .......... .......... .......... .......... ..........  0%  239M 28m41s
   350K .......... .......... .......... .......... ..........  0%  106M 25m7s
   400K .......... .......... .......... .......... ..........  0% 1.26M 23m39s
   450K .......... .......... .......... .......... ..........  0%  406K 25m5s
   500K .......... .......... .......... .......... ..........  0% 87.0M 22m49s
   550K .......... .......... .......... .......... ..........  0% 91.6M 20m56s
   600K .......... .......... .......... ..

## Adding Read Groups and Marking Duplicates
- Read Groups (@RG tags) identify the sample name, sequencing platform, and library. We will use GATK's AddOrReplaceReadGroups and sort the file by genomic coordinates.

- During PCR amplification in sequencing, the exact same DNA fragment can be sequenced multiple times. We need to flag these "artifacts" so they don't skew our variant calling statistics.

In [6]:
%%bash
# --- 1. PROCESS THE NORMAL SAMPLE ---
echo "🔧 Preprocessing NORMAL tissue..."
java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar AddOrReplaceReadGroups \
    -I /content/WES-LUNG/NORMAL.bam \
    -O /content/NORMAL.rg.bam \
    -RGID 1 -RGLB WES_Lib -RGPL ILLUMINA -RGPU unit1 -RGSM NORMAL \
    --CREATE_INDEX true

java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar MarkDuplicates \
    -I /content/NORMAL.rg.bam \
    -O /content/NORMAL.marked_dups.bam \
    -M /content/normal_metrics.txt \
    --CREATE_INDEX true

# --- 2. PROCESS THE TUMOR SAMPLE ---
echo "🔧 Preprocessing TUMOR tissue..."
java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar AddOrReplaceReadGroups \
    -I /content/WES-LUNG/TUMOR.bam \
    -O /content/TUMOR.rg.bam \
    -RGID 2 -RGLB WES_Lib -RGPL ILLUMINA -RGPU unit2 -RGSM TUMOR \
    --CREATE_INDEX true

java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar MarkDuplicates \
    -I /content/TUMOR.rg.bam \
    -O /content/TUMOR.marked_dups.bam \
    -M /content/tumor_metrics.txt \
    --CREATE_INDEX true

echo "✅ Preprocessing complete! Both BAMs are ready."

🔧 Preprocessing NORMAL tissue...
Tool returned:
0
Tool returned:
0
🔧 Preprocessing TUMOR tissue...
Tool returned:
0
Tool returned:
0
✅ Preprocessing complete! Both BAMs are ready.


09:14:52.696 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
[Mon Jun 01 09:14:52 UTC 2026] AddOrReplaceReadGroups --INPUT /content/WES-LUNG/NORMAL.bam --OUTPUT /content/NORMAL.rg.bam --RGID 1 --RGLB WES_Lib --RGPL ILLUMINA --RGPU unit1 --RGSM NORMAL --CREATE_INDEX true --VERBOSITY INFO --QUIET false --VALIDATION_STRINGENCY STRICT --COMPRESSION_LEVEL 2 --MAX_RECORDS_IN_RAM 500000 --CREATE_MD5_FILE false --help false --version false --showHidden false --USE_JDK_DEFLATER false --USE_JDK_INFLATER false
[Mon Jun 01 09:14:53 UTC 2026] Executing as root@02be900f4b77 on Linux 6.6.122+ amd64; OpenJDK 64-Bit Server VM 17.0.18+8-Ubuntu-122.04.1; Deflater: Intel; Inflater: Intel; Provider GCS is available; Picard version: Version:4.6.2.0
INFO	2026-06-01 09:14:53	AddOrReplaceReadGroups	Created read-group ID=1 PL=ILLUMINA LB=WES_Lib SM=NORMAL

[Mon Jun 01 09:14:54 UTC 2026] picar

Base Quality Score Recalibration (BQSR)

The sequencing machine often introduces systematic errors when assigning quality scores to bases. BQSR uses a database of known polymorphic sites (like dbSNP) to adjust these quality scores so they reflect the true error probability.

⚠️ As my data is 0.1% subsample, BQSR might struggle or throw warnings due to low data volume, For BQSR, we need to run it like this:

In [7]:
# 1. Download known sites (dbSNP for hg19)
# !wget ftp://ftp.ncbi.nih.gov/snp/organisms/human_9606_b151_GRCh37p13/VCF/All_20180418.vcf.gz

# 2. Build the recalibration table
# !./gatk-4.6.2.0/gatk BaseRecalibrator \
 #    -I /content/WES-LUNG_dedup.bam \
  #   -R hg19.fa \
   #  --known-sites All_20180418.vcf.gz \
    # -O /content/recal_data.table

# 3. Apply the recalibration to the BAM
# !./gatk-4.6.2.0/gatk ApplyBQSR \
  #   -I /content/WES-LUNG_dedup.bam \
   #  -R hg19.fa \
    # --bqsr-recal-file /content/recal_data.table \
    # -O /content/WES-LUNG_final.bam

## Germline Variant Discovery

In [8]:
!java -Xmx4g -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar HaplotypeCaller \
    -R /content/reference_hg19/hg19.fa \
    -I /content/NORMAL.marked_dups.bam \
    -O /content/germline_raw.vcf

09:15:26.259 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
09:15:26.499 INFO  HaplotypeCaller - ------------------------------------------------------------
09:15:26.506 INFO  HaplotypeCaller - The Genome Analysis Toolkit (GATK) v4.6.2.0
09:15:26.506 INFO  HaplotypeCaller - For support and documentation go to https://software.broadinstitute.org/gatk/
09:15:26.506 INFO  HaplotypeCaller - Executing as root@02be900f4b77 on Linux v6.6.122+ amd64
09:15:26.506 INFO  HaplotypeCaller - Java runtime: OpenJDK 64-Bit Server VM v17.0.18+8-Ubuntu-122.04.1
09:15:26.507 INFO  HaplotypeCaller - Start Date/Time: June 1, 2026 at 9:15:26 AM UTC
09:15:26.507 INFO  HaplotypeCaller - ------------------------------------------------------------
09:15:26.507 INFO  HaplotypeCaller - ------------------------------------------------------------
09:15:26.508 INFO  HaplotypeCaller - HTSJDK Ver

## Variant Filtration

In [9]:
%%bash
echo "🧹 Applying relaxed filters for downsampled data..."
java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar VariantFiltration \
    -R /content/reference_hg19/hg19.fa \
    -V /content/germline_raw.vcf \
    -filter "QUAL < 10.0" --filter-name "LowQual" \
    -O /content/germline_filtered_relaxed.vcf

# Isolate header metadata and passing variants
grep -E '^#|PASS' /content/germline_filtered_relaxed.vcf > /content/germline_final_passed.vcf

echo "📊 High-Confidence Germline Variants Remaining:"
grep -v '^#' /content/germline_final_passed.vcf | wc -l

🧹 Applying relaxed filters for downsampled data...
📊 High-Confidence Germline Variants Remaining:
175


09:27:18.834 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
09:27:19.108 INFO  VariantFiltration - ------------------------------------------------------------
09:27:19.114 INFO  VariantFiltration - The Genome Analysis Toolkit (GATK) v4.6.2.0
09:27:19.114 INFO  VariantFiltration - For support and documentation go to https://software.broadinstitute.org/gatk/
09:27:19.115 INFO  VariantFiltration - Executing as root@02be900f4b77 on Linux v6.6.122+ amd64
09:27:19.115 INFO  VariantFiltration - Java runtime: OpenJDK 64-Bit Server VM v17.0.18+8-Ubuntu-122.04.1
09:27:19.115 INFO  VariantFiltration - Start Date/Time: June 1, 2026 at 9:27:18 AM UTC
09:27:19.115 INFO  VariantFiltration - ------------------------------------------------------------
09:27:19.115 INFO  VariantFiltration - ------------------------------------------------------------
09:27:19.117 INFO  VariantFiltr

## Functional Annotation via SnpEff

In [10]:
# Install and Setup SnpEff

%%bash
cd /content/
wget -q --show-progress https://downloads.sourceforge.net/project/snpeff/snpEff_latest_core.zip
unzip -qo snpEff_latest_core.zip
echo "📊 Verifying jar placement:"
ls -lh /content/snpEff/snpEff.jar

📊 Verifying jar placement:
-rw-rw-r-- 1 root root 21M Nov 24  2017 /content/snpEff/snpEff.jar



     0K .......... .......... .......... .......... ..........  0% 7.55M 9s
    50K .......... .......... .......... .......... ..........  0% 4.07M 13s
   100K .......... .......... .......... .......... ..........  0% 90.2M 9s
   150K .......... .......... .......... .......... ..........  0% 29.3M 7s
   200K .......... .......... .......... .......... ..........  0% 3.80M 9s
   250K .......... .......... .......... .......... ..........  0%  103M 8s
   300K .......... .......... .......... .......... ..........  0%  109M 7s
   350K .......... .......... .......... .......... ..........  0% 16.5M 6s
   400K .......... .......... .......... .......... ..........  0% 82.8M 6s
   450K .......... .......... .......... .......... ..........  0%  108M 5s
   500K .......... .......... .......... .......... ..........  0% 65.4M 5s
   550K .......... .......... .......... .......... ..........  0% 56.5M 5s
   600K .......... .......... .......... .......... ..........  0%  238M 4s
   650K ..

In [11]:
# Functional Annotation of Variants Using SnpEff

!java -Xmx4g -jar /content/snpEff/snpEff.jar \
    hg19 \
    /content/germline_final_passed.vcf \
    > /content/germline_annotated.vcf

## Inspect Variants

In [12]:
import pandas as pd

vcf_path = "/content/germline_annotated.vcf"
germline_variants = []

with open(vcf_path, 'r') as f:
    for line in f:
        if line.startswith('#'):
            continue
        chunks = line.strip().split('\t')
        info = chunks[7]

        if "ANN=" in info:
            ann_field = [x for x in info.split(';') if x.startswith("ANN=")][0]
            first_effect = ann_field.replace("ANN=", "").split(',')[0].split('|')

            gene_name = first_effect[3]
            effect = first_effect[1]
            impact = first_effect[2]

            germline_variants.append([chunks[0], chunks[1], chunks[3], chunks[4], gene_name, effect, impact])

df_germline = pd.DataFrame(germline_variants, columns=['Chrom', 'Position', 'Ref', 'Alt', 'Gene', 'Effect', 'Impact'])

print("=== COMPLETE GERMLINE VARIANT PROFILE (ALL IMPACT LEVELS) ===")
if not df_germline.empty:
    # Display the top 20 variants to see what HaplotypeCaller captured
    print(df_germline.head(20).to_string(index=False))
    print(f"\n📊 Total background variants found in this slice: {len(df_germline)}")
else:
    print("The variant file is completely empty. Double-check if 'germline_final_passed.vcf' contains variants.")

=== COMPLETE GERMLINE VARIANT PROFILE (ALL IMPACT LEVELS) ===
Chrom Position Ref       Alt    Gene              Effect   Impact
chr11 26692481   C         T SLC5A12 3_prime_UTR_variant MODIFIER
chr11 26692594   T TTGTGTGTG SLC5A12 3_prime_UTR_variant MODIFIER
chr11 26692631   A         G SLC5A12 3_prime_UTR_variant MODIFIER
chr11 26692733   T         C SLC5A12  synonymous_variant      LOW
chr11 26692742   G         A SLC5A12  synonymous_variant      LOW
chr11 26692811   C         G SLC5A12      intron_variant MODIFIER
chr11 26692933   A         T SLC5A12      intron_variant MODIFIER
chr11 26692967   C         T SLC5A12      intron_variant MODIFIER
chr11 26694979   A         G SLC5A12  synonymous_variant      LOW
chr11 26702664   C         T SLC5A12  synonymous_variant      LOW
chr11 26705310   A         G SLC5A12  synonymous_variant      LOW
chr11 26707847   G      GTTC SLC5A12      intron_variant MODIFIER
chr11 26724972   T     TACAC SLC5A12      intron_variant MODIFIER
chr11 26724995

## Extract SNP and INDELS

In [13]:
%%bash
# 1. Extract only the Single Nucleotide Polymorphisms (SNPs)
java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar SelectVariants \
    -V /content/germline_final_passed.vcf \
    -select-type SNP \
    -O /content/germline_snps.vcf

# 2. Extract only the Insertions and Deletions (Indels)
java -jar /content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar SelectVariants \
    -V /content/germline_final_passed.vcf \
    -select-type INDEL \
    -O /content/germline_indels.vcf

echo "📊 Quick Count Breakdown:"
echo -n "Total SNPs: " && grep -v '^#' /content/germline_snps.vcf | wc -l
echo -n "Total Indels: " && grep -v '^#' /content/germline_indels.vcf | wc -l

📊 Quick Count Breakdown:
Total SNPs: 158
Total Indels: 17


09:28:50.702 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
09:28:50.942 INFO  SelectVariants - ------------------------------------------------------------
09:28:50.947 INFO  SelectVariants - The Genome Analysis Toolkit (GATK) v4.6.2.0
09:28:50.948 INFO  SelectVariants - For support and documentation go to https://software.broadinstitute.org/gatk/
09:28:50.948 INFO  SelectVariants - Executing as root@02be900f4b77 on Linux v6.6.122+ amd64
09:28:50.948 INFO  SelectVariants - Java runtime: OpenJDK 64-Bit Server VM v17.0.18+8-Ubuntu-122.04.1
09:28:50.949 INFO  SelectVariants - Start Date/Time: June 1, 2026 at 9:28:50 AM UTC
09:28:50.949 INFO  SelectVariants - ------------------------------------------------------------
09:28:50.949 INFO  SelectVariants - ------------------------------------------------------------
09:28:50.951 INFO  SelectVariants - HTSJDK Version: 4.2

In [14]:
# Annotate SNPs
!java -Xmx4g -jar /content/snpEff/snpEff.jar hg19 /content/germline_snps.vcf > /content/germline_snps_annotated.vcf

# Annotate Indels
!java -Xmx4g -jar /content/snpEff/snpEff.jar hg19 /content/germline_indels.vcf > /content/germline_indels_annotated.vcf

In [15]:
import pandas as pd

def parse_vcf_to_df(vcf_path):
    variants = []
    with open(vcf_path, 'r') as f:
        for line in f:
            if line.startswith('#'): continue
            chunks = line.strip().split('\t')
            info = chunks[7]
            if "ANN=" in info:
                ann_field = [x for x in info.split(';') if x.startswith("ANN=")][0]
                first_effect = ann_field.replace("ANN=", "").split(',')[0].split('|')
                gene_name = first_effect[3]
                effect = first_effect[1]
                impact = first_effect[2]
                variants.append([chunks[0], chunks[1], chunks[3], chunks[4], gene_name, effect, impact])
    return pd.DataFrame(variants, columns=['Chrom', 'Position', 'Ref', 'Alt', 'Gene', 'Effect', 'Impact'])

# Generate individual dataframes
df_snps = parse_vcf_to_df("/content/germline_snps_annotated.vcf")
df_indels = parse_vcf_to_df("/content/germline_indels_annotated.vcf")

# --- DISPLAY RESULTS ---
print(f"=== GERMLINE SNPs DISCOVERED ({len(df_snps)} total) ===")
print(df_snps.head(10).to_string(index=False))

print("\n" + "="*60 + "\n")

print(f"=== GERMLINE INDELS DISCOVERED ({len(df_indels)} total) ===")
print(df_indels.head(10).to_string(index=False))

# Optional: Export them as clean spreadsheets!
df_snps.to_csv("/content/germline_snps_summary.csv", index=False)
df_indels.to_csv("/content/germline_indels_summary.csv", index=False)
print("\n💾 Saved summaries to 'germline_snps_summary.csv' and 'germline_indels_summary.csv'!")

=== GERMLINE SNPs DISCOVERED (158 total) ===
Chrom Position Ref Alt    Gene              Effect   Impact
chr11 26692481   C   T SLC5A12 3_prime_UTR_variant MODIFIER
chr11 26692631   A   G SLC5A12 3_prime_UTR_variant MODIFIER
chr11 26692733   T   C SLC5A12  synonymous_variant      LOW
chr11 26692742   G   A SLC5A12  synonymous_variant      LOW
chr11 26692811   C   G SLC5A12      intron_variant MODIFIER
chr11 26692933   A   T SLC5A12      intron_variant MODIFIER
chr11 26692967   C   T SLC5A12      intron_variant MODIFIER
chr11 26694979   A   G SLC5A12  synonymous_variant      LOW
chr11 26702664   C   T SLC5A12  synonymous_variant      LOW
chr11 26705310   A   G SLC5A12  synonymous_variant      LOW


=== GERMLINE INDELS DISCOVERED (17 total) ===
Chrom  Position   Ref       Alt    Gene                         Effect   Impact
chr11  26692594     T TTGTGTGTG SLC5A12            3_prime_UTR_variant MODIFIER
chr11  26707847     G      GTTC SLC5A12                 intron_variant MODIFIER
chr11  

## Conclusion and Key Findings

In this study, germline variant calling was performed using a standard GATK-based pipeline followed by functional annotation with SnpEff.

A total of **158 SNPs** and **17 INDELs** were identified from the input BAM-derived VCF files. The majority of variants were located in **non-coding regions**, including intronic and 3′ untranslated regions (3′ UTRs), and were classified as **MODIFIER or LOW impact**, suggesting limited predicted functional consequence.

Notably, most variants were mapped to genes such as *SLC5A12*, with a high proportion of synonymous and intronic substitutions, indicating strong evolutionary conservation of coding regions in this locus. A small number of INDELs showed **MODERATE impact**, including an in-frame insertion in *FAM83G*, which may warrant further functional investigation.

Overall, this dataset reflects a typical germline variant profile dominated by non-coding and low-impact changes. These results provide a foundation for downstream analyses such as population comparison, disease association studies, or integration with transcriptomic data.

The processed results were successfully exported as:
- `germline_snps_summary.csv`
- `germline_indels_summary.csv`